In [41]:
# Scientific Computing
import pandas as pd
import numpy as np

# Text processing.
import re 
import ftfy
import spacy
from nltk import ngrams

# Initializing tok2vec for lemmatization.
# See "Natural Language Processing in Action" for code reference. 
nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "attribute_ruler", "ner"])

In [42]:
Survey_df = pd.read_csv("../Clean_Data_Resources/Survey_Results.csv")

C:\Users\clara\AppData\Local\Temp\ipykernel_39700\3248790261.py:1: DtypeWarning: Columns (0: Course_Code, 1: R_Dropin_Tutoring, 2: R_Online_Active_Participation, 3: R_Online_Engaging_Content, 4: R_Weekly_Emails, 5: T_Online_Experience_Suggestions, 6: T_Other_Suggestions) have mixed types. Specify dtype option on import or set low_memory=False.
  Survey_df = pd.read_csv("../Clean_Data_Resources/Survey_Results.csv")


In [43]:
# Define all survey columns that are made of text.
Text_Cols = [
    "T_Collaboration_Understanding",
    "T_Leader_Performance_Suggestions",
    "T_Other_Suggestions",
    "T_Program_Overall_Suggestions",
]

In [44]:
# Tokenize responses.

# Filter out responses that are non-responses. So NA, none, ect.
Non_Responses = {"n/a", "na", "none", "nothing", "no", "n"}

def tokenize_response(text):
    # Guard against NaN and empty strings.
    if not isinstance(text, str) or text.strip() == "":
        return []
    # Guard against non-responses.
    if text.strip().lower().replace("/", "") in Non_Responses:
        return []
    text = text.replace("\u2019", "'").replace("\u2018", "'") # Unicode encodings for apostrophes. Not scary.
    text = ftfy.fix_text(text)
    doc = nlp(text.lower())
    # Strip punctuation, whitespace, contractions, and emails.
    return [tok.text for tok in doc
            if not tok.is_punct
            and not tok.is_space
            and tok.text not in {"n't", "ca", "m", "'m", "ve", "'ve", "re", "'re"}
            and not re.match(r'\S+@\S+', tok.text)]

In [45]:
# Use an apply function to tokenize all columns in Text_Cols:
for col in Text_Cols:
    Survey_df[col + "_tokens"] = Survey_df[col].apply(tokenize_response)

c:\Documents\GitHub_Repos\SI_ML_Survey\si_text_env\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)
c:\Documents\GitHub_Repos\SI_ML_Survey\si_text_env\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)
c:\Documents\GitHub_Repos\SI_ML_Survey\si_text_env\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger

In [46]:
# Function to enact lemmatization.
def lemmatize_response(tokens):
    if not tokens:
        return []
    doc = nlp(" ".join(tokens))
    return [tok.lemma_ for tok in doc if not tok.is_punct and not tok.is_space]

# Function to enact ngrams, wher n specifies the n...grams. 
def make_ngrams(tokens, n):
    return [" ".join(gram) for gram in ngrams(tokens, n)]

In [47]:
# Lemmatize, then build bigrams and trigrams for each column.
for col in Text_Cols:
    lemma_col = col + "_lemmas"
    Survey_df[lemma_col] = Survey_df[col + "_tokens"].apply(lemmatize_response)
    Survey_df[col + "_bigrams"]  = Survey_df[lemma_col].apply(lambda x: make_ngrams(x, 2))
    Survey_df[col + "_trigrams"] = Survey_df[lemma_col].apply(lambda x: make_ngrams(x, 3))